In [ ]:
import os
from alb_sim.config.run import RunConfig
from alb_sim.config.sea_floor import SeaFloorConfig
from alb_sim.config.sea_surface import SeaSurfaceConfig
from alb_sim.config.water import TurbidityLayerConfig, WaterConfig, FournierForandConfig
from alb_sim.config.simulation import SimulationConfig
from alb_sim.config.sensor import SensorConfig
from alb_sim.execution.parallel import run_parallel

In [ ]:
simulation_config = SimulationConfig(
    water=WaterConfig(
        layers=(
            TurbidityLayerConfig(
                height=1.5, absorption_coefficient=0.05, scattering_coefficient=4
            ),
            TurbidityLayerConfig(
                height=0.25, absorption_coefficient=0, scattering_coefficient=1, fournier_forand_parameters=FournierForandConfig(refractive_index_ratio=1.15, junge_slope=4.065)
            ),
            TurbidityLayerConfig(
                height=0.25, absorption_coefficient=0, scattering_coefficient=1.5, fournier_forand_parameters=FournierForandConfig(refractive_index_ratio=1.15, junge_slope=4.065)
            ),
            TurbidityLayerConfig(
                height=0.25, absorption_coefficient=0, scattering_coefficient=2.0, fournier_forand_parameters=FournierForandConfig(refractive_index_ratio=1.15, junge_slope=4.065)
            ),
            TurbidityLayerConfig(
                height=0.25, absorption_coefficient=0, scattering_coefficient=2.5, fournier_forand_parameters=FournierForandConfig(refractive_index_ratio=1.15, junge_slope=4.065)
            ),
            TurbidityLayerConfig(
                height=4.5, absorption_coefficient=0.005, scattering_coefficient=4
            ),
        )
    ),
    sensor=SensorConfig(sample_rate=2e9),
    sea_floor=SeaFloorConfig(albedo=0.1),
    sea_surface=SeaSurfaceConfig(albedo=0.1, roughness=0.05)
)
run_config = RunConfig(
    processes=16,
    batches_forward=(16) * 15,
    batches_backward=(16) * 5,
)

In [ ]:
simulation_config = SimulationConfig(
    water=WaterConfig(
        layers=(
            TurbidityLayerConfig(
                height=0.5, absorption_coefficient=0.15, scattering_coefficient=0.5, fournier_forand_parameters=FournierForandConfig(refractive_index_ratio=1.10, junge_slope=4.065)
            ),
            TurbidityLayerConfig(
                height=0.5, absorption_coefficient=0.15, scattering_coefficient=0.75, fournier_forand_parameters=FournierForandConfig(refractive_index_ratio=1.10, junge_slope=4.065)
            ),
            TurbidityLayerConfig(
                height=6, absorption_coefficient=0.15, scattering_coefficient=3, fournier_forand_parameters=FournierForandConfig(refractive_index_ratio=1.08, junge_slope=3.483)
            ),
        )
    ),
    sensor=SensorConfig(sample_rate=2e9),
    sea_floor=SeaFloorConfig(albedo=0.05),
    sea_surface=SeaSurfaceConfig(albedo=0.1, roughness=0.05)
)
run_config = RunConfig(
    processes=16,
    batches_forward=(16) * 15,
    batches_backward=(16) * 2,
)

In [ ]:
waveform = run_parallel(simulation_config, run_config)

data = waveform

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.arange(len(next(iter(data.values()))))
labels = [pt.name for pt in data.keys()]
values = np.vstack(list(data.values()))
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(1850, 2050)
ax.stackplot(x, values, labels=labels)
ax.legend(loc="upper right")
ax.set_xlabel("Step / Distance")
ax.set_ylabel("Photon contribution")
ax.set_title("Photon contributions (stacked)")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.sum(list(data.values()), axis=0))
ax.set_xlim(1850, 2050)
ax.set_xlabel("Step / Distance")
ax.set_ylabel("Photon contribution")
ax.set_title("Photon contributions")
plt.tight_layout()
plt.show()

In [ ]:
import pickle
with open(r"waveform.pickle", "wb") as output_file:
    pickle.dump(waveform, output_file)